In [51]:
import os
import warnings
import numpy as np
import pandas as pd
import torch

from pytorch_forecasting import (
    TimeSeriesDataSet,
    TemporalFusionTransformer
)

from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

from lightning.pytorch import Trainer, seed_everything
from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    LearningRateMonitor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

warnings.filterwarnings("ignore")

seed_everything(42, workers=True)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Seed set to 42


PyTorch: 2.14.0+cpu
CUDA available: False


In [53]:
PROJECT_DIR = r"C:/Users/307124/Documents/Mutual-Fund-Disclosure-Intelligence"

DATA_FILE = os.path.join(
    PROJECT_DIR,
    "data",
    "processed",
    "final",
    "stock_macro_monthly_target.csv"
)

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models",
    "tft_v2"
)

os.makedirs(MODEL_DIR, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Data:", DATA_FILE)
print("Model directory:", MODEL_DIR)

assert os.path.exists(DATA_FILE), f"Dataset not found: {DATA_FILE}"

Project: C:/Users/307124/Documents/Mutual-Fund-Disclosure-Intelligence
Data: C:/Users/307124/Documents/Mutual-Fund-Disclosure-Intelligence\data\processed\final\stock_macro_monthly_target.csv
Model directory: C:/Users/307124/Documents/Mutual-Fund-Disclosure-Intelligence\models\tft_v2


In [54]:
df = pd.read_csv(DATA_FILE)

print("=" * 100)
print("DATASET")
print("=" * 100)

print("Shape:", df.shape)

print("\nColumns:")
for i, col in enumerate(df.columns):
    print(f"{i:2d} : {col}")

print("\nFirst 5 rows:")
display(df.head())

DATASET
Shape: (60846, 23)

Columns:
 0 : isin
 1 : symbol
 2 : monthly_open
 3 : monthly_high
 4 : monthly_low
 5 : monthly_close
 6 : monthly_adj_close
 7 : monthly_volume
 8 : trading_days
 9 : date
10 : stock_return_1m
11 : oil_price_usd_bbl
12 : gold_price_inr_10g
13 : cpi_index
14 : cpi_inflation_yoy_pct
15 : oil_return_1m_pct
16 : oil_return_12m_pct
17 : gold_return_1m_pct
18 : gold_return_12m_pct
19 : inflation_change_1m
20 : house_price_index
21 : hpi_growth_12m_pct
22 : target_return_1m

First 5 rows:


,isin,symbol,monthly_open,monthly_high,monthly_low,monthly_close,monthly_adj_close,monthly_volume,trading_days,date,...,cpi_index,cpi_inflation_yoy_pct,oil_return_1m_pct,oil_return_12m_pct,gold_return_1m_pct,gold_return_12m_pct,inflation_change_1m,house_price_index,hpi_growth_12m_pct,target_return_1m
0,INE002A01018,RELIANCE,249.407104,262.847809,232.720535,239.143723,211.659332,642993345.0,19,2010-01-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-6.428039
1,INE002A01018,RELIANCE,237.155045,241.463837,219.245560,223.771500,198.053787,324665427.0,19,2010-02-01,...,NaN,NaN,8.578222,NaN,3.192272,NaN,NaN,NaN,NaN,9.734929
2,INE002A01018,RELIANCE,225.154434,253.955917,223.874359,245.555481,217.334183,419629083.0,21,2010-03-01,...,NaN,NaN,6.585901,NaN,-3.187257,NaN,NaN,NaN,NaN,-3.784027
3,INE002A01018,RELIANCE,246.412659,262.802094,231.326172,236.263580,209.110199,376994477.0,20,2010-04-01,...,NaN,NaN,5.731567,NaN,4.698833,NaN,NaN,NaN,NaN,1.909299
4,INE002A01018,RELIANCE,234.526337,249.932846,223.097183,239.006577,213.102737,475654929.0,21,2010-05-01,...,NaN,NaN,-15.347673,NaN,7.613534,NaN,NaN,NaN,NaN,4.232022


In [55]:
print("=" * 100)
print("DATA TYPES")
print("=" * 100)

print(df.dtypes)

DATA TYPES
isin                         str
symbol                       str
monthly_open             float64
monthly_high             float64
monthly_low              float64
monthly_close            float64
monthly_adj_close        float64
monthly_volume           float64
trading_days               int64
date                         str
stock_return_1m          float64
oil_price_usd_bbl        float64
gold_price_inr_10g       float64
cpi_index                float64
cpi_inflation_yoy_pct    float64
oil_return_1m_pct        float64
oil_return_12m_pct       float64
gold_return_1m_pct       float64
gold_return_12m_pct      float64
inflation_change_1m      float64
house_price_index        float64
hpi_growth_12m_pct       float64
target_return_1m         float64
dtype: object


In [56]:
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["isin", "date"]
).reset_index(drop=True)

print("Date range:")
print("Start:", df["date"].min())
print("End  :", df["date"].max())

print("\nNumber of stocks:", df["isin"].nunique())
print("Number of rows:", len(df))

Date range:
Start: 2010-01-01 00:00:00
End  : 2026-09-01 00:00:00

Number of stocks: 437
Number of rows: 60846


In [58]:
# CELL 6 — TARGET CHECK AND CLEANING

TARGET_COLUMN = "target_return_1m"

print("=" * 100)
print("TARGET CHECK")
print("=" * 100)

target = df[TARGET_COLUMN]

print(target.describe())

missing_target = target.isna().sum()
infinite_target = np.isinf(target).sum()

print("\nMissing:", missing_target)
print("Infinite:", infinite_target)

# The final month of each stock has no next-month price,
# so target_return_1m is naturally unavailable there.

if missing_target > 0:

    missing_target_rows = df[target.isna()].copy()

    print("\nMissing-target rows:", len(missing_target_rows))
    print(
        "Stocks affected:",
        missing_target_rows["isin"].nunique()
    )

    print("\nExample missing-target rows:")
    display(
        missing_target_rows[
            ["isin", "symbol", "date", "monthly_adj_close"]
        ].head(20)
    )

    # Verify that these are the final observations for each stock
    last_dates = (
        df.groupby("isin")["date"]
        .max()
        .rename("last_stock_date")
    )

    missing_target_rows = missing_target_rows.merge(
        last_dates,
        on="isin",
        how="left"
    )

    not_last_month = (
        missing_target_rows["date"]
        != missing_target_rows["last_stock_date"]
    ).sum()

    print(
        "\nMissing targets that are NOT the final observation "
        f"of their stock: {not_last_month}"
    )

    assert not_last_month == 0, (
        "Some target NaNs occur before the final observation "
        "of a stock. Investigate the target construction."
    )

    # Remove only rows where the target cannot exist
    df = df[
        df[TARGET_COLUMN].notna()
    ].copy()

    print(
        "\nRemoved final observation rows:",
        missing_target
    )

print("\nRemaining rows:", len(df))
print("Remaining target NaNs:", df[TARGET_COLUMN].isna().sum())
print("Remaining target Inf:", np.isinf(df[TARGET_COLUMN]).sum())

assert df[TARGET_COLUMN].isna().sum() == 0
assert np.isinf(df[TARGET_COLUMN]).sum() == 0

print("\n✅ Target is now complete and finite.")

TARGET CHECK
count    60409.000000
mean         1.788788
std         11.206155
min        -82.461538
25%         -4.705887
50%          0.924257
75%          7.349875
max        350.303410
Name: target_return_1m, dtype: float64

Missing: 437
Infinite: 0

Missing-target rows: 437
Stocks affected: 437

Example missing-target rows:


,isin,symbol,date,monthly_adj_close
200,INE002A01018,RELIANCE,2026-09-01,1226.400024
401,INE003A01024,SIEMENS,2026-09-01,3913.899902
421,INE004A01022,PROTEAN,2026-09-01,490.299988
622,INE007A01025,CRISIL,2026-09-01,4577.500000
823,INE009A01021,INFY,2026-09-01,1051.400024
846,INE00H001014,SWIGGY,2026-09-01,273.049988
848,INE00IT01020,MILKYMIST,2026-09-01,274.070007
915,INE00LO01017,CRAFTSMAN,2026-09-01,11138.000000
926,INE00Q601028,STUDDS,2026-09-01,407.100006
1020,INE00R701025,DALBHARAT,2026-09-01,1761.400024



Missing targets that are NOT the final observation of their stock: 0

Removed final observation rows: 437

Remaining rows: 60409
Remaining target NaNs: 0
Remaining target Inf: 0

✅ Target is now complete and finite.


In [59]:
check = df[
    ["isin", "date", "monthly_adj_close", TARGET_COLUMN]
].copy()

check["next_adj_close"] = (
    check.groupby("isin")["monthly_adj_close"]
    .shift(-1)
)

check["calculated_target"] = (
    check["next_adj_close"] /
    check["monthly_adj_close"] - 1
) * 100

comparison = check.dropna(
    subset=["calculated_target", TARGET_COLUMN]
).copy()

difference = (
    comparison[TARGET_COLUMN] -
    comparison["calculated_target"]
).abs()

print("Rows checked:", len(comparison))
print("Maximum difference:", difference.max())
print("Mean difference:", difference.mean())

assert difference.max() < 1e-6

print("\n✅ Target formula verified.")

Rows checked: 59972
Maximum difference: 6.750155989720952e-14
Mean difference: 2.8462532957790004e-15

✅ Target formula verified.


In [60]:
print("=" * 100)
print("NUMERIC FEATURE QUALITY")
print("=" * 100)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns

quality_report = []

for col in numeric_cols:
    quality_report.append({
        "column": col,
        "NaN": int(df[col].isna().sum()),
        "Inf": int(np.isinf(df[col].to_numpy()).sum()),
        "min": df[col].min(),
        "max": df[col].max()
    })

quality_df = pd.DataFrame(quality_report)

display(quality_df)

NUMERIC FEATURE QUALITY


,column,NaN,Inf,min,max
0,monthly_open,0,0,1.540000,5.670000e+04
1,monthly_high,0,0,2.490000,5.999400e+04
2,monthly_low,0,0,1.245000,5.295415e+04
3,monthly_close,0,0,1.470000,5.671120e+04
4,monthly_adj_close,0,0,1.442670,5.629013e+04
5,monthly_volume,0,0,0.000000,5.321441e+09
6,trading_days,0,0,1.000000,2.300000e+01
7,stock_return_1m,210,0,-82.461538,5.315915e+02
8,oil_price_usd_bbl,0,0,25.270000,1.258900e+02
9,gold_price_inr_10g,0,0,16035.455348,1.412787e+05


In [61]:
HPI_COLUMNS = [
    "house_price_index",
    "hpi_growth_12m_pct"
]

print("=" * 100)
print("HPI MISSING VALUES")
print("=" * 100)

for col in HPI_COLUMNS:
    print(
        f"{col:25} "
        f"NaN = {df[col].isna().sum():8d} "
        f"({df[col].isna().mean()*100:.2f}%)"
    )

HPI MISSING VALUES
house_price_index         NaN =    40966 (67.81%)
hpi_growth_12m_pct        NaN =    45220 (74.86%)


In [62]:
hpi_missing_by_month = (
    df.groupby("date")[HPI_COLUMNS]
    .apply(lambda x: x.isna().mean() * 100)
)

display(hpi_missing_by_month.head(20))

,house_price_index,hpi_growth_12m_pct
date,,
2010-01-01,100.0,100.0
2010-02-01,100.0,100.0
2010-03-01,100.0,100.0
2010-04-01,100.0,100.0
2010-05-01,100.0,100.0
2010-06-01,100.0,100.0
2010-07-01,100.0,100.0
2010-08-01,100.0,100.0
2010-09-01,100.0,100.0


In [63]:
MACRO_COLUMNS = [
    "oil_price_usd_bbl",
    "gold_price_inr_10g",
    "cpi_index",
    "cpi_inflation_yoy_pct",
    "oil_return_1m_pct",
    "oil_return_12m_pct",
    "gold_return_1m_pct",
    "gold_return_12m_pct",
    "inflation_change_1m",
    "house_price_index",
    "hpi_growth_12m_pct"
]

macro_monthly = (
    df[["date"] + MACRO_COLUMNS]
    .drop_duplicates("date")
    .sort_values("date")
    .reset_index(drop=True)
)

macro_monthly[MACRO_COLUMNS] = (
    macro_monthly[MACRO_COLUMNS]
    .ffill()
    .bfill()
)

print("Remaining macro NaNs:")

print(
    macro_monthly[MACRO_COLUMNS]
    .isna()
    .sum()
)

Remaining macro NaNs:
oil_price_usd_bbl        0
gold_price_inr_10g       0
cpi_index                0
cpi_inflation_yoy_pct    0
oil_return_1m_pct        0
oil_return_12m_pct       0
gold_return_1m_pct       0
gold_return_12m_pct      0
inflation_change_1m      0
house_price_index        0
hpi_growth_12m_pct       0
dtype: int64


In [64]:
df = df.drop(
    columns=MACRO_COLUMNS
).merge(
    macro_monthly,
    on="date",
    how="left"
)

df = df.sort_values(
    ["isin", "date"]
).reset_index(drop=True)

print("Dataset shape:", df.shape)

print("\nRemaining macro NaNs:")

print(
    df[MACRO_COLUMNS]
    .isna()
    .sum()
)

Dataset shape: (60409, 23)

Remaining macro NaNs:
oil_price_usd_bbl        0
gold_price_inr_10g       0
cpi_index                0
cpi_inflation_yoy_pct    0
oil_return_1m_pct        0
oil_return_12m_pct       0
gold_return_1m_pct       0
gold_return_12m_pct      0
inflation_change_1m      0
house_price_index        0
hpi_growth_12m_pct       0
dtype: int64


In [65]:
stock_history = (
    df.groupby("isin")
    .agg(
        observations=("date", "count"),
        first_date=("date", "min"),
        last_date=("date", "max")
    )
    .sort_values("observations")
)

print("=" * 100)
print("STOCK HISTORY")
print("=" * 100)

print("Number of stocks:", len(stock_history))

display(stock_history.head(30))

print("\nObservation statistics:")
print(stock_history["observations"].describe())

STOCK HISTORY
Number of stocks: 437


,observations,first_date,last_date
isin,,,
INE05C901015,1,2026-08-01,2026-08-01
INE0FOO01011,1,2026-08-01,2026-08-01
INE1KZI01025,1,2026-08-01,2026-08-01
INE16W401027,1,2026-08-01,2026-08-01
INE00IT01020,1,2026-08-01,2026-08-01
INE0QJW01029,2,2026-07-01,2026-08-01
INE0ISX01025,2,2026-07-01,2026-08-01
INE640G01020,2,2026-07-01,2026-08-01
INE084101034,2,2026-07-01,2026-08-01



Observation statistics:
count    437.000000
mean     138.235698
std       74.734925
min        1.000000
25%       61.000000
50%      200.000000
75%      200.000000
max      200.000000
Name: observations, dtype: float64


In [66]:
MIN_OBSERVATIONS = 36

eligible_isins = stock_history[
    stock_history["observations"] >= MIN_OBSERVATIONS
].index

print("Minimum observations:", MIN_OBSERVATIONS)
print("Eligible stocks:", len(eligible_isins))
print("Removed stocks:", len(stock_history) - len(eligible_isins))

Minimum observations: 36
Eligible stocks: 369
Removed stocks: 68


In [67]:
model_df = df[
    df["isin"].isin(eligible_isins)
].copy()

model_df = model_df.sort_values(
    ["isin", "date"]
).reset_index(drop=True)

print("=" * 100)
print("TFT MODEL UNIVERSE")
print("=" * 100)

print("Rows:", len(model_df))
print("Stocks:", model_df["isin"].nunique())
print("Date range:", model_df["date"].min(), "to", model_df["date"].max())

TFT MODEL UNIVERSE
Rows: 59434
Stocks: 369
Date range: 2010-01-01 00:00:00 to 2026-08-01 00:00:00


In [68]:
all_months = np.sort(
    model_df["date"].unique()
)

month_to_idx = {
    month: idx
    for idx, month in enumerate(all_months)
}

model_df["time_idx"] = model_df["date"].map(
    month_to_idx
).astype(int)

model_df["month"] = (
    model_df["date"]
    .dt.month
    .astype(int)
)

print("time_idx:")
print(
    model_df[
        ["date", "time_idx"]
    ].drop_duplicates().head(15)
)

print("\nTotal months:", model_df["time_idx"].nunique())
print(
    "time_idx range:",
    model_df["time_idx"].min(),
    "to",
    model_df["time_idx"].max()
)

time_idx:
         date  time_idx
0  2010-01-01         0
1  2010-02-01         1
2  2010-03-01         2
3  2010-04-01         3
4  2010-05-01         4
5  2010-06-01         5
6  2010-07-01         6
7  2010-08-01         7
8  2010-09-01         8
9  2010-10-01         9
10 2010-11-01        10
11 2010-12-01        11
12 2011-01-01        12
13 2011-02-01        13
14 2011-03-01        14

Total months: 200
time_idx range: 0 to 199


In [69]:
STATIC_CATEGORICALS = [
    "isin"
]

TIME_VARYING_KNOWN_CATEGORICALS = [
]

TIME_VARYING_KNOWN_REALS = [
    "time_idx",
    "month"
]

TIME_VARYING_UNKNOWN_REALS = [
    "monthly_open",
    "monthly_high",
    "monthly_low",
    "monthly_close",
    "monthly_adj_close",
    "monthly_volume",
    "trading_days",
    "stock_return_1m",
    "oil_price_usd_bbl",
    "gold_price_inr_10g",
    "cpi_index",
    "cpi_inflation_yoy_pct",
    "oil_return_1m_pct",
    "oil_return_12m_pct",
    "gold_return_1m_pct",
    "gold_return_12m_pct",
    "inflation_change_1m",
    "house_price_index",
    "hpi_growth_12m_pct"
]

print("Static categorical:", STATIC_CATEGORICALS)
print("Known reals:", TIME_VARYING_KNOWN_REALS)
print("Unknown reals:", len(TIME_VARYING_UNKNOWN_REALS))

Static categorical: ['isin']
Known reals: ['time_idx', 'month']
Unknown reals: 19


In [73]:
# CELL 18 — CLEAN TFT INPUTS

print("=" * 100)
print("CLEANING TFT INPUT DATA")
print("=" * 100)

print("Rows before cleaning:", len(model_df))

# ---------------------------------------------------------------------
# 1. Check stock_return_1m
# ---------------------------------------------------------------------

missing_stock_return = model_df["stock_return_1m"].isna().sum()

print("\nMissing stock_return_1m:", missing_stock_return)

if missing_stock_return > 0:

    print("\nThese rows cannot have a valid 1-month historical return")
    print("because they do not have a previous monthly observation.")

    display(
        model_df.loc[
            model_df["stock_return_1m"].isna(),
            [
                "isin",
                "symbol",
                "date",
                "monthly_adj_close",
                "stock_return_1m"
            ]
        ].head(20)
    )

    # Remove only rows where the previous-month return is unavailable
    model_df = model_df[
        model_df["stock_return_1m"].notna()
    ].copy()

    print(
        "\nRemoved rows with missing stock_return_1m:",
        missing_stock_return
    )


# ---------------------------------------------------------------------
# 2. Check ALL TFT inputs again
# ---------------------------------------------------------------------

TFT_INPUT_COLUMNS = (
    STATIC_CATEGORICALS
    + KNOWN_CATEGORICALS
    + KNOWN_REALS
    + UNKNOWN_REALS
    + [TARGET_COLUMN]
)

TFT_INPUT_COLUMNS = list(dict.fromkeys(TFT_INPUT_COLUMNS))


quality_after = []

for col in TFT_INPUT_COLUMNS:

    if col not in model_df.columns:

        quality_after.append({
            "column": col,
            "NaN": -1,
            "Inf": -1
        })

        continue

    series = model_df[col]

    nan_count = int(series.isna().sum())

    if pd.api.types.is_numeric_dtype(series):

        inf_count = int(
            np.isinf(
                series.to_numpy(dtype=float)
            ).sum()
        )

    else:

        inf_count = 0

    quality_after.append({
        "column": col,
        "NaN": nan_count,
        "Inf": inf_count
    })


quality_after = pd.DataFrame(quality_after)

display(
    quality_after[
        (quality_after["NaN"] > 0) |
        (quality_after["Inf"] > 0) |
        (quality_after["NaN"] == -1)
    ]
)


# ---------------------------------------------------------------------
# 3. Final assertions
# ---------------------------------------------------------------------

total_nan = quality_after.loc[
    quality_after["NaN"] > 0,
    "NaN"
].sum()

total_inf = quality_after.loc[
    quality_after["Inf"] > 0,
    "Inf"
].sum()

missing_columns = quality_after[
    quality_after["NaN"] == -1
]


print("\nRows after cleaning:", len(model_df))
print("Total TFT NaNs:", total_nan)
print("Total TFT Infs:", total_inf)
print("Missing TFT columns:", len(missing_columns))


assert total_nan == 0
assert total_inf == 0
assert len(missing_columns) == 0

print("\n✅ ALL TFT INPUTS ARE FINITE AND PRESENT.")

CLEANING TFT INPUT DATA
Rows before cleaning: 59434

Missing stock_return_1m: 142

These rows cannot have a valid 1-month historical return
because they do not have a previous monthly observation.


,isin,symbol,date,monthly_adj_close,stock_return_1m
800,INE00LO01017,CRAFTSMAN,2021-03-01,1406.736206,NaN
866,INE00R701025,DALBHARAT,2018-12-01,1067.883667,NaN
959,INE00WC01027,AFFLE,2019-08-01,167.860001,NaN
1444,INE010J01012,TEJASNET,2017-06-01,299.243408,NaN
2355,INE018E01016,SBICARD,2020-03-01,607.367432,NaN
2633,INE01QM01018,EMUDHRA,2022-06-01,249.694122,NaN
3084,INE022Q01020,IEX,2017-10-01,45.757252,NaN
3191,INE027H01010,MAXHEALTH,2020-08-01,105.729919,NaN
4864,INE041025011,EMBASSY,2019-04-01,204.346024,NaN
5353,INE045J01034,FCL,2015-01-01,2.470323,NaN



Removed rows with missing stock_return_1m: 142


,column,NaN,Inf



Rows after cleaning: 59292
Total TFT NaNs: 0
Total TFT Infs: 0
Missing TFT columns: 0

✅ ALL TFT INPUTS ARE FINITE AND PRESENT.


In [74]:
# CELL 19 — FINAL MODEL DATA PREPARATION

model_df = model_df.sort_values(
    ["isin", "date"]
).reset_index(drop=True)

# Recreate global time index
unique_dates = sorted(model_df["date"].unique())

date_to_idx = {
    date: idx
    for idx, date in enumerate(unique_dates)
}

model_df["time_idx"] = model_df["date"].map(date_to_idx).astype(int)

print("=" * 100)
print("FINAL MODEL DATA")
print("=" * 100)

print("Rows:", len(model_df))
print("Stocks:", model_df["isin"].nunique())
print("Dates:", model_df["date"].nunique())

print(
    "Date range:",
    model_df["date"].min(),
    "to",
    model_df["date"].max()
)

print(
    "time_idx range:",
    model_df["time_idx"].min(),
    "to",
    model_df["time_idx"].max()
)

print("\nTarget statistics:")
display(model_df[TARGET_COLUMN].describe())

print("\n✅ Model data prepared.")

FINAL MODEL DATA
Rows: 59292
Stocks: 369
Dates: 200
Date range: 2010-01-01 00:00:00 to 2026-08-01 00:00:00
time_idx range: 0 to 199

Target statistics:


count    59292.000000
mean         1.795816
std         11.191105
min        -82.461538
25%         -4.682282
50%          0.938460
75%          7.346827
max        350.303410
Name: target_return_1m, dtype: float64


✅ Model data prepared.


In [75]:
# CELL 20 — STOCK HISTORY CHECK

stock_history = (
    model_df.groupby("isin")
    .size()
    .reset_index(name="observations")
)

print("=" * 100)
print("STOCK HISTORY")
print("=" * 100)

display(stock_history["observations"].describe())

print("\nStocks with < 12 observations:",
      (stock_history["observations"] < 12).sum())

print("Stocks with < 24 observations:",
      (stock_history["observations"] < 24).sum())

print("Stocks with < 36 observations:",
      (stock_history["observations"] < 36).sum())

print("Stocks with >= 36 observations:",
      (stock_history["observations"] >= 36).sum())

STOCK HISTORY


count    369.000000
mean     160.682927
std       57.354919
min       35.000000
25%      107.000000
50%      200.000000
75%      200.000000
max      200.000000
Name: observations, dtype: float64


Stocks with < 12 observations: 0
Stocks with < 24 observations: 0
Stocks with < 36 observations: 1
Stocks with >= 36 observations: 368


In [76]:
# CELL 21 — FILTER STOCKS WITH SUFFICIENT HISTORY

MIN_OBSERVATIONS = 36

eligible_isins = stock_history.loc[
    stock_history["observations"] >= MIN_OBSERVATIONS,
    "isin"
]

print("Eligible stocks:", len(eligible_isins))
print("Total stocks before:", model_df["isin"].nunique())

model_df = model_df[
    model_df["isin"].isin(eligible_isins)
].copy()

model_df = model_df.sort_values(
    ["isin", "date"]
).reset_index(drop=True)

print("Total stocks after:", model_df["isin"].nunique())
print("Rows after filtering:", len(model_df))

assert model_df["isin"].nunique() == len(eligible_isins)

print("\n✅ Stock-history filtering complete.")

Eligible stocks: 368
Total stocks before: 369
Total stocks after: 368
Rows after filtering: 59257

✅ Stock-history filtering complete.


In [77]:
# CELL 22 — CHRONOLOGICAL TRAIN / VALIDATION SPLIT

TRAIN_RATIO = 0.80

unique_time_idx = sorted(
    model_df["time_idx"].unique()
)

split_position = int(
    len(unique_time_idx) * TRAIN_RATIO
)

training_cutoff = unique_time_idx[
    split_position - 1
]

print("=" * 100)
print("TRAIN / VALIDATION SPLIT")
print("=" * 100)

print("Training cutoff:", training_cutoff)

train_df = model_df[
    model_df["time_idx"] <= training_cutoff
].copy()

validation_df = model_df[
    model_df["time_idx"] > training_cutoff
].copy()

print("\nTraining:")
print("Rows:", len(train_df))
print("Stocks:", train_df["isin"].nunique())

print("\nValidation:")
print("Rows:", len(validation_df))
print("Stocks:", validation_df["isin"].nunique())

print("\nTraining target:")
display(train_df[TARGET_COLUMN].describe())

print("\nValidation target:")
display(validation_df[TARGET_COLUMN].describe())

TRAIN / VALIDATION SPLIT
Training cutoff: 159

Training:
Rows: 44578
Stocks: 361

Validation:
Rows: 14679
Stocks: 368

Training target:


count    44578.000000
mean         1.838260
std         11.396075
min        -68.726833
25%         -4.693397
50%          0.992430
75%          7.443650
max        350.303410
Name: target_return_1m, dtype: float64


Validation target:


count    14679.000000
mean         1.665593
std         10.534428
min        -82.461538
25%         -4.633888
50%          0.765108
75%          7.103518
max        110.087713
Name: target_return_1m, dtype: float64

In [78]:
# CELL 23 — KEEP ONLY STOCKS SEEN DURING TRAINING

train_isins = set(
    train_df["isin"].unique()
)

validation_df = validation_df[
    validation_df["isin"].isin(train_isins)
].copy()

validation_df = validation_df.sort_values(
    ["isin", "date"]
).reset_index(drop=True)

print("=" * 100)
print("VALIDATION UNIVERSE")
print("=" * 100)

print("Training stocks:", len(train_isins))
print("Validation stocks:", validation_df["isin"].nunique())
print("Validation rows:", len(validation_df))

unseen = set(
    validation_df["isin"].unique()
) - train_isins

print("Unseen validation stocks:", len(unseen))

assert len(unseen) == 0

print("\n✅ Validation universe is compatible with training.")

VALIDATION UNIVERSE
Training stocks: 361
Validation stocks: 361
Validation rows: 14413
Unseen validation stocks: 0

✅ Validation universe is compatible with training.


In [79]:
# CELL 24 — TFT FEATURE CONFIGURATION

STATIC_CATEGORICALS = [
    "isin"
]

KNOWN_CATEGORICALS = []

KNOWN_REALS = [
    "time_idx",
    "month"
]

UNKNOWN_REALS = [
    "monthly_open",
    "monthly_high",
    "monthly_low",
    "monthly_close",
    "monthly_adj_close",
    "monthly_volume",
    "trading_days",
    "stock_return_1m",

    "oil_price_usd_bbl",
    "gold_price_inr_10g",
    "cpi_index",
    "cpi_inflation_yoy_pct",
    "oil_return_1m_pct",
    "oil_return_12m_pct",
    "gold_return_1m_pct",
    "gold_return_12m_pct",
    "inflation_change_1m",
    "house_price_index",
    "hpi_growth_12m_pct"
]

TARGET_COLUMN = "target_return_1m"

MAX_ENCODER_LENGTH = 24
MAX_PREDICTION_LENGTH = 1
MIN_ENCODER_LENGTH = 12

BATCH_SIZE = 128

print("Static categoricals:", STATIC_CATEGORICALS)
print("Known reals:", KNOWN_REALS)
print("Unknown reals:", len(UNKNOWN_REALS))
print("Target:", TARGET_COLUMN)
print("Encoder length:", MAX_ENCODER_LENGTH)
print("Prediction length:", MAX_PREDICTION_LENGTH)

Static categoricals: ['isin']
Known reals: ['time_idx', 'month']
Unknown reals: 19
Target: target_return_1m
Encoder length: 24
Prediction length: 1


In [80]:
# CELL 25 — CREATE TRAINING DATASET

from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer

training_dataset = TimeSeriesDataSet(

    train_df,

    time_idx="time_idx",

    target=TARGET_COLUMN,

    group_ids=["isin"],

    min_encoder_length=MIN_ENCODER_LENGTH,
    max_encoder_length=MAX_ENCODER_LENGTH,

    min_prediction_length=1,
    max_prediction_length=MAX_PREDICTION_LENGTH,

    static_categoricals=STATIC_CATEGORICALS,

    time_varying_known_categoricals=KNOWN_CATEGORICALS,

    time_varying_known_reals=KNOWN_REALS,

    time_varying_unknown_reals=UNKNOWN_REALS,

    target_normalizer=GroupNormalizer(
        method="standard",
        groups=["isin"],
        center=True,
        scale_by_group=False,
        transformation=None
    ),

    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,

    allow_missing_timesteps=True
)

print("=" * 100)
print("TRAINING DATASET")
print("=" * 100)

print("Number of training samples:",
      len(training_dataset))

print("\n✅ TimeSeriesDataSet created.")

TRAINING DATASET
Number of training samples: 44297

✅ TimeSeriesDataSet created.


In [81]:
# CELL 26 — CREATE VALIDATION DATASET

validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    validation_df,
    min_prediction_idx=training_cutoff + 1,
    stop_randomization=True
)

print("=" * 100)
print("VALIDATION DATASET")
print("=" * 100)

print("Number of validation samples:",
      len(validation_dataset))

print("\n✅ Validation dataset created.")

VALIDATION DATASET
Number of validation samples: 14400

✅ Validation dataset created.


In [82]:
# CELL 27 — CREATE DATALOADERS

train_dataloader = training_dataset.to_dataloader(
    train=True,
    batch_size=BATCH_SIZE,
    num_workers=0
)

validation_dataloader = validation_dataset.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0
)

print("=" * 100)
print("DATALOADERS")
print("=" * 100)

print("Training batches:",
      len(train_dataloader))

print("Validation batches:",
      len(validation_dataloader))

print("\n✅ DataLoaders ready.")

DATALOADERS
Training batches: 346
Validation batches: 113

✅ DataLoaders ready.


In [83]:
# CELL 28 — TRAINING BATCH FINITE CHECK

batch = next(iter(train_dataloader))

x, y = batch

print("=" * 100)
print("TRAINING BATCH CHECK")
print("=" * 100)

print("Encoder continuous shape:",
      x["encoder_cont"].shape)

print("Decoder continuous shape:",
      x["decoder_cont"].shape)

print("Encoder categorical shape:",
      x["encoder_cat"].shape)

print("Target shape:",
      y[0].shape if isinstance(y, tuple) else y.shape)


def check_tensor(name, tensor):

    tensor = tensor.float()

    nan_count = torch.isnan(tensor).sum().item()
    inf_count = torch.isinf(tensor).sum().item()

    print(
        f"{name}: "
        f"NaN={nan_count}, "
        f"Inf={inf_count}, "
        f"min={tensor.nan_to_num().min().item():.4f}, "
        f"max={tensor.nan_to_num().max().item():.4f}"
    )

    return nan_count, inf_count


import torch

enc_nan, enc_inf = check_tensor(
    "encoder_cont",
    x["encoder_cont"]
)

dec_nan, dec_inf = check_tensor(
    "decoder_cont",
    x["decoder_cont"]
)

target_tensor = y[0] if isinstance(y, tuple) else y

target_nan, target_inf = check_tensor(
    "target",
    target_tensor
)

assert enc_nan == 0
assert enc_inf == 0

assert dec_nan == 0
assert dec_inf == 0

assert target_nan == 0
assert target_inf == 0

print("\n✅ First training batch is completely finite.")

TRAINING BATCH CHECK
Encoder continuous shape: torch.Size([128, 24, 25])
Decoder continuous shape: torch.Size([128, 1, 25])
Encoder categorical shape: torch.Size([128, 24, 1])
Target shape: torch.Size([128, 1])
encoder_cont: NaN=0, Inf=0, min=-4.6280, max=10.1849
decoder_cont: NaN=0, Inf=0, min=-3.6965, max=6.0490
target: NaN=0, Inf=0, min=-26.8800, max=37.7193

✅ First training batch is completely finite.


In [84]:
# CELL 29 — FULL VALIDATION BATCH FINITE CHECK

print("=" * 100)
print("FULL VALIDATION BATCH FINITE CHECK")
print("=" * 100)

bad_batches = []

for batch_idx, batch in enumerate(validation_dataloader):

    x, y = batch

    tensors_to_check = {
        "encoder_cont": x["encoder_cont"],
        "decoder_cont": x["decoder_cont"],
        "encoder_cat": x["encoder_cat"],
        "target": y[0] if isinstance(y, tuple) else y
    }

    batch_bad = False

    for name, tensor in tensors_to_check.items():

        tensor_float = tensor.float()

        nan_count = torch.isnan(tensor_float).sum().item()
        inf_count = torch.isinf(tensor_float).sum().item()

        if nan_count > 0 or inf_count > 0:

            batch_bad = True

            bad_batches.append({
                "batch": batch_idx,
                "tensor": name,
                "NaN": nan_count,
                "Inf": inf_count
            })

    if batch_bad:
        print(f"⚠️ Bad batch detected: {batch_idx}")


print("\n" + "=" * 100)
print("VALIDATION RESULT")
print("=" * 100)

if len(bad_batches) == 0:

    print("Bad batches:", 0)
    print("\n✅ ALL VALIDATION BATCHES ARE FINITE.")

else:

    bad_df = pd.DataFrame(bad_batches)

    print("Bad batches:", bad_df["batch"].nunique())
    print("Total bad tensor entries:", len(bad_df))

    display(bad_df)

    raise ValueError(
        "Validation DataLoader contains NaN/Inf values."
    )

FULL VALIDATION BATCH FINITE CHECK

VALIDATION RESULT
Bad batches: 0

✅ ALL VALIDATION BATCHES ARE FINITE.


In [85]:
# CELL 30 — CREATE FRESH TFT MODEL

import torch
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

print("=" * 100)
print("CREATING FRESH TEMPORAL FUSION TRANSFORMER")
print("=" * 100)

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,

    # Safer learning rate
    learning_rate=0.0003,

    # Model architecture
    hidden_size=64,
    attention_head_size=4,
    hidden_continuous_size=32,
    dropout=0.1,

    # Forecasting loss
    loss=QuantileLoss(),

    # 7 quantiles:
    # 0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98
    output_size=7,

    log_interval=10,
    reduce_on_plateau_patience=4
)

print("\nModel created successfully.")

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in tft.parameters()
        if p.requires_grad
    )
)

print("\nLearning rate:", 0.0003)
print("Hidden size:", 64)
print("Attention heads:", 4)
print("Dropout:", 0.1)
print("\n✅ Fresh TFT created.")

CREATING FRESH TEMPORAL FUSION TRANSFORMER

Model created successfully.
Trainable parameters: 430850

Learning rate: 0.0003
Hidden size: 64
Attention heads: 4
Dropout: 0.1

✅ Fresh TFT created.


In [86]:
# CELL 31 — TFT PARAMETER FINITE CHECK

print("=" * 100)
print("TFT PARAMETER CHECK")
print("=" * 100)

bad_parameters = []

for name, parameter in tft.named_parameters():

    values = parameter.detach()

    nan_count = torch.isnan(values).sum().item()
    inf_count = torch.isinf(values).sum().item()

    if nan_count > 0 or inf_count > 0:

        bad_parameters.append({
            "parameter": name,
            "NaN": nan_count,
            "Inf": inf_count
        })


if len(bad_parameters) == 0:

    print("Bad parameters:", 0)
    print("\n✅ ALL MODEL PARAMETERS ARE FINITE.")

else:

    bad_parameter_df = pd.DataFrame(bad_parameters)

    display(bad_parameter_df)

    raise ValueError(
        "Fresh TFT contains NaN/Inf parameters."
    )

TFT PARAMETER CHECK
Bad parameters: 0

✅ ALL MODEL PARAMETERS ARE FINITE.


In [87]:
# CELL 32 — FRESH TFT FORWARD PASS CHECK

print("=" * 100)
print("FRESH TFT FORWARD PASS CHECK")
print("=" * 100)

# Get a fresh batch
batch = next(iter(train_dataloader))

x, y = batch

# Put model in evaluation mode
tft.eval()

with torch.no_grad():

    output = tft(x)

# TFT output object contains prediction tensor
prediction = output["prediction"]

print("Prediction shape:", prediction.shape)

nan_count = torch.isnan(prediction).sum().item()
inf_count = torch.isinf(prediction).sum().item()

print("Prediction NaN:", nan_count)
print("Prediction Inf:", inf_count)

finite_prediction = prediction[
    torch.isfinite(prediction)
]

if len(finite_prediction) > 0:

    print(
        "Prediction min:",
        finite_prediction.min().item()
    )

    print(
        "Prediction max:",
        finite_prediction.max().item()
    )

    print(
        "Prediction mean:",
        finite_prediction.mean().item()
    )


assert nan_count == 0
assert inf_count == 0

print("\n✅ FRESH TFT FORWARD PASS IS COMPLETELY FINITE.")

FRESH TFT FORWARD PASS CHECK
Prediction shape: torch.Size([128, 1, 7])
Prediction NaN: 0
Prediction Inf: 0
Prediction min: -21.849597930908203
Prediction max: 25.717355728149414
Prediction mean: 0.5798865556716919

✅ FRESH TFT FORWARD PASS IS COMPLETELY FINITE.


In [93]:
# CELL 30 — CREATE FRESH TFT MODEL
# Interpretation logging disabled to avoid the PyTorch Forecasting
# 'interpretation' KeyError during epoch-end validation logging.

import torch
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

print("=" * 100)
print("CREATING FRESH TEMPORAL FUSION TRANSFORMER")
print("=" * 100)

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,

    learning_rate=0.0003,

    hidden_size=64,
    attention_head_size=4,
    hidden_continuous_size=32,
    dropout=0.1,

    loss=QuantileLoss(),
    output_size=7,

    # IMPORTANT:
    # Disable interpretation logging during training.
    log_interval=-1,

    reduce_on_plateau_patience=4
)

print("\nModel created successfully.")

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in tft.parameters()
        if p.requires_grad
    )
)

print("\nLearning rate:", 0.0003)
print("Hidden size:", 64)
print("Attention heads:", 4)
print("Dropout:", 0.1)
print("Interpretation logging: DISABLED")

print("\n✅ Fresh TFT created.")

CREATING FRESH TEMPORAL FUSION TRANSFORMER

Model created successfully.
Trainable parameters: 430850

Learning rate: 0.0003
Hidden size: 64
Attention heads: 4
Dropout: 0.1
Interpretation logging: DISABLED

✅ Fresh TFT created.


In [94]:
# CELL 31 — TFT PARAMETER FINITE CHECK

print("=" * 100)
print("TFT PARAMETER CHECK")
print("=" * 100)

bad_parameters = []

for name, parameter in tft.named_parameters():

    values = parameter.detach()

    nan_count = torch.isnan(values).sum().item()
    inf_count = torch.isinf(values).sum().item()

    if nan_count > 0 or inf_count > 0:

        bad_parameters.append({
            "parameter": name,
            "NaN": nan_count,
            "Inf": inf_count
        })


if len(bad_parameters) == 0:

    print("Bad parameters:", 0)
    print("\n✅ ALL MODEL PARAMETERS ARE FINITE.")

else:

    bad_parameter_df = pd.DataFrame(bad_parameters)

    display(bad_parameter_df)

    raise ValueError(
        "Fresh TFT contains NaN/Inf parameters."
    )

TFT PARAMETER CHECK
Bad parameters: 0

✅ ALL MODEL PARAMETERS ARE FINITE.


In [95]:
# CELL 32 — FRESH TFT FORWARD PASS CHECK

print("=" * 100)
print("FRESH TFT FORWARD PASS CHECK")
print("=" * 100)

batch = next(iter(train_dataloader))

x, y = batch

tft.eval()

with torch.no_grad():
    output = tft(x)

prediction = output["prediction"]

print("Prediction shape:", prediction.shape)

nan_count = torch.isnan(prediction).sum().item()
inf_count = torch.isinf(prediction).sum().item()

print("Prediction NaN:", nan_count)
print("Prediction Inf:", inf_count)

finite_prediction = prediction[
    torch.isfinite(prediction)
]

if len(finite_prediction) > 0:

    print(
        "Prediction min:",
        finite_prediction.min().item()
    )

    print(
        "Prediction max:",
        finite_prediction.max().item()
    )

    print(
        "Prediction mean:",
        finite_prediction.mean().item()
    )


assert nan_count == 0
assert inf_count == 0

print("\n✅ FRESH TFT FORWARD PASS IS COMPLETELY FINITE.")

FRESH TFT FORWARD PASS CHECK
Prediction shape: torch.Size([128, 1, 7])
Prediction NaN: 0
Prediction Inf: 0
Prediction min: -30.47005271911621
Prediction max: 25.647037506103516
Prediction mean: 1.9914861917495728

✅ FRESH TFT FORWARD PASS IS COMPLETELY FINITE.


In [96]:
# CELL 33 — CONFIGURE TFT TRAINING

from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    LearningRateMonitor
)
import lightning.pytorch as pl
import os

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "models",
    "tft"
)

os.makedirs(MODEL_DIR, exist_ok=True)

early_stop_callback = EarlyStopping(
    monitor="val_loss",
    min_delta=0.001,
    patience=5,
    verbose=True,
    mode="min"
)

checkpoint_callback = ModelCheckpoint(
    dirpath=MODEL_DIR,
    filename="tft-best-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss",
    mode="min",
    save_top_k=1
)

lr_monitor = LearningRateMonitor(
    logging_interval="epoch"
)

print("=" * 100)
print("TFT TRAINING CONFIGURATION")
print("=" * 100)

print("Model directory:", MODEL_DIR)
print("Max epochs:", 30)
print("Early stopping patience:", 5)
print("Learning rate:", 0.0003)
print("Batch size:", BATCH_SIZE)

print("\n✅ Training callbacks configured.")

TFT TRAINING CONFIGURATION
Model directory: c:\Users\307124\Documents\Mutual-Fund-Disclosure-Intelligence\models\tft
Max epochs: 30
Early stopping patience: 5
Learning rate: 0.0003
Batch size: 128

✅ Training callbacks configured.


In [97]:
# CELL 34 — TRAIN TFT

trainer = pl.Trainer(
    max_epochs=30,

    accelerator="auto",
    devices=1,

    gradient_clip_val=0.1,

    callbacks=[
        early_stop_callback,
        checkpoint_callback,
        lr_monitor
    ],

    enable_progress_bar=True,
    enable_model_summary=True,

    log_every_n_steps=10,

    default_root_dir=MODEL_DIR
)

print("=" * 100)
print("STARTING TFT TRAINING")
print("=" * 100)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=validation_dataloader
)

print("\n" + "=" * 100)
print("TRAINING FINISHED")
print("=" * 100)

print("Best checkpoint:")
print(checkpoint_callback.best_model_path)

print("Best validation loss:")
print(checkpoint_callback.best_model_score)

print("\n✅ TFT training completed.")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


STARTING TFT TRAINING


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ eval │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ eval │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │ 15.5 K │ eval │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │  1.6 K │ eval │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │ 21.0 K │ eval │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  166 K │ eval │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 20.7 K │ eval │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │ 16.8 K │ eval │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │ 16.8 K │ eval │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │ 16.8 K │ eval │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │ 16.8 K │ eval │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │ 33.3 K │ eval │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │ 33.3 K │ eval │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  8.3 K │ eval │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │    128 │ eval │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │ 20.9 K │ eval │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │ 10.4 K │ eval │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  8.4 K │ eval │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │ 16.8 K │ eval │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  8.4 K │ eval │     0 │
│ 20 │ output_layer                       │ Linear                          │    455 │ eval │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴──────┴───────┘

Trainable params: 430 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 430 K                                                                                                
Total estimated model params size (MB): 1.723                                                                      
Modules in train mode: 0                                                                                           
Modules in eval mode: 583                                                                                          
Total FLOPs: 0

Output()

Metric val_loss improved. New best score: 277781.969


Metric val_loss improved by 0.188 >= min_delta = 0.001. New best score: 277781.781


Monitored metric val_loss did not improve in the last 5 records. Best score: 277781.781. Signaling Trainer to stop.



TRAINING FINISHED
Best checkpoint:
C:\Users\307124\Documents\Mutual-Fund-Disclosure-Intelligence\models\tft\tft-best-epoch=01-val_loss=277781.7812.ckpt
Best validation loss:
tensor(277781.7812)

✅ TFT training completed.


In [101]:
# CELL 35 — LOAD BEST TFT CHECKPOINT + FULL VALIDATION EVALUATION
# FIXED: Rebuild validation dataset from the ORIGINAL training_dataset
# to guarantee identical categorical encoders.

import os
import re
import numpy as np
import pandas as pd
import torch

from pytorch_forecasting import TemporalFusionTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("=" * 100)
print("TFT CHECKPOINT VALIDATION")
print("=" * 100)


# =========================================================================
# 1. FIND BEST CHECKPOINT
# =========================================================================

checkpoint_dir = (
    r"C:\Users\307124\Documents"
    r"\Mutual-Fund-Disclosure-Intelligence"
    r"\models\tft"
)

checkpoint_files = [
    os.path.join(checkpoint_dir, f)
    for f in os.listdir(checkpoint_dir)
    if f.endswith(".ckpt")
]

if not checkpoint_files:
    raise FileNotFoundError(
        f"No checkpoint files found in:\n{checkpoint_dir}"
    )


def extract_val_loss(path):

    filename = os.path.basename(path)

    match = re.search(
        r"val_loss=([0-9]+(?:\.[0-9]+)?)\.ckpt$",
        filename
    )

    if match:
        return float(match.group(1))

    return float("inf")


print("\nAvailable checkpoints:")

for path in checkpoint_files:
    print(
        f" - {os.path.basename(path)}"
        f" | val_loss={extract_val_loss(path)}"
    )


best_checkpoint = min(
    checkpoint_files,
    key=extract_val_loss
)

best_checkpoint_loss = extract_val_loss(
    best_checkpoint
)

print("\n" + "-" * 100)
print("BEST CHECKPOINT")
print("-" * 100)

print("File:")
print(os.path.basename(best_checkpoint))

print(
    f"Validation loss: {best_checkpoint_loss:.6f}"
)


# =========================================================================
# 2. LOAD BEST TFT
# =========================================================================

print("\nLoading checkpoint...")

best_tft = TemporalFusionTransformer.load_from_checkpoint(
    best_checkpoint
)

best_tft.eval()

print("✅ Checkpoint loaded.")

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in best_tft.parameters()
        if p.requires_grad
    )
)


# =========================================================================
# 3. REBUILD VALIDATION DATASET FROM training_dataset
# =========================================================================
#
# IMPORTANT:
#
# Do NOT use a separately constructed validation_dataset here.
#
# from_dataset() copies the categorical encoders/scalers from
# training_dataset, ensuring that the ISIN category indices match
# the embedding layer stored in the checkpoint.
# =========================================================================

print("\nRebuilding validation dataset from training_dataset...")

validation_dataset_eval = TimeSeriesDataSet.from_dataset(
    training_dataset,
    validation_df,
    predict=False,
    stop_randomization=True
)

print("✅ Validation dataset rebuilt.")

print(
    "Validation samples:",
    len(validation_dataset_eval)
)


# =========================================================================
# 4. VALIDATION DATALOADER
# =========================================================================

validation_loader = validation_dataset_eval.to_dataloader(
    train=False,
    batch_size=BATCH_SIZE,
    num_workers=0
)

print(
    "Validation batches:",
    len(validation_loader)
)


# =========================================================================
# 5. CHECK CATEGORICAL INDICES BEFORE FORWARD PASS
# =========================================================================

print("\nChecking categorical indices...")

sample_batch = next(iter(validation_loader))

sample_x, sample_y = sample_batch

# Move sample batch to model device
device = next(best_tft.parameters()).device

sample_x_device = {
    key: value.to(device)
    if torch.is_tensor(value)
    else value
    for key, value in sample_x.items()
}

# ISIN categorical tensor
if "encoder_cat" in sample_x_device:

    encoder_cat = sample_x_device["encoder_cat"]

    print(
        "encoder_cat shape:",
        encoder_cat.shape
    )

    print(
        "encoder_cat min:",
        encoder_cat.min().item()
    )

    print(
        "encoder_cat max:",
        encoder_cat.max().item()
    )


# =========================================================================
# 6. CHECK EMBEDDING SIZE
# =========================================================================

print("\nChecking TFT categorical embedding...")

try:

    isin_embedding = (
        best_tft.input_embeddings["isin"]
    )

    embedding_size = (
        isin_embedding.embedding.num_embeddings
    )

    print(
        "ISIN embedding categories:",
        embedding_size
    )

except Exception as e:

    print(
        "Could not directly inspect ISIN embedding:"
    )

    print(e)


# =========================================================================
# 7. FULL VALIDATION PREDICTION
# =========================================================================

print("\n" + "=" * 100)
print("RUNNING FULL VALIDATION PREDICTION")
print("=" * 100)

all_predictions = []
all_actuals = []

bad_batches = []

with torch.no_grad():

    for batch_idx, batch in enumerate(
        validation_loader
    ):

        x, y = batch

        # -------------------------------------------------------------
        # Move tensors to model device
        # -------------------------------------------------------------

        x = {
            key: value.to(device)
            if torch.is_tensor(value)
            else value
            for key, value in x.items()
        }

        # -------------------------------------------------------------
        # Target
        # -------------------------------------------------------------

        if isinstance(y, (tuple, list)):
            target = y[0]
        else:
            target = y

        target = target.to(device)

        # -------------------------------------------------------------
        # Check categorical indices
        # -------------------------------------------------------------

        if "encoder_cat" in x:

            encoder_cat = x["encoder_cat"]

            if torch.any(encoder_cat < 0):

                raise ValueError(
                    f"Negative categorical index "
                    f"found in batch {batch_idx}"
                )

        # -------------------------------------------------------------
        # Forward pass
        # -------------------------------------------------------------

        try:

            output = best_tft(x)

        except Exception as e:

            print(
                f"\n❌ Forward pass failed "
                f"at batch {batch_idx}"
            )

            print(
                "Error:",
                repr(e)
            )

            print(
                "\nThis batch will be inspected."
            )

            print(
                "encoder_cat shape:",
                x["encoder_cat"].shape
            )

            print(
                "encoder_cat min:",
                x["encoder_cat"].min().item()
            )

            print(
                "encoder_cat max:",
                x["encoder_cat"].max().item()
            )

            raise

        # -------------------------------------------------------------
        # Extract prediction
        # -------------------------------------------------------------

        if hasattr(
            output,
            "prediction"
        ):

            prediction = output.prediction

        elif (
            isinstance(output, dict)
            and "prediction" in output
        ):

            prediction = output["prediction"]

        else:

            prediction = output

        # -------------------------------------------------------------
        # Quantile prediction
        #
        # TFT output:
        # [batch, prediction_length, quantiles]
        #
        # Use 0.5 / median quantile.
        # -------------------------------------------------------------

        if prediction.ndim == 3:

            quantiles = best_tft.loss.quantiles

            median_idx = min(
                range(len(quantiles)),
                key=lambda i:
                    abs(
                        float(
                            quantiles[i]
                        ) - 0.5
                    )
            )

            prediction_point = (
                prediction[
                    :,
                    :,
                    median_idx
                ]
            )

        elif prediction.ndim == 2:

            prediction_point = prediction

        else:

            prediction_point = (
                prediction.reshape(-1, 1)
            )

        # -------------------------------------------------------------
        # Flatten
        # -------------------------------------------------------------

        prediction_np = (
            prediction_point
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )

        actual_np = (
            target
            .detach()
            .cpu()
            .numpy()
            .reshape(-1)
        )

        all_predictions.extend(
            prediction_np
        )

        all_actuals.extend(
            actual_np
        )

        # -------------------------------------------------------------
        # Progress
        # -------------------------------------------------------------

        if (
            (batch_idx + 1) % 25 == 0
            or
            batch_idx == len(
                validation_loader
            ) - 1
        ):

            print(
                f"Processed "
                f"{batch_idx + 1}/"
                f"{len(validation_loader)}"
            )


print(
    "\n✅ Full validation prediction completed."
)


# =========================================================================
# 8. CONVERT TO NUMPY
# =========================================================================

predictions = np.asarray(
    all_predictions,
    dtype=np.float64
)

actuals = np.asarray(
    all_actuals,
    dtype=np.float64
)

print("\nPrediction count:", len(predictions))
print("Actual count:", len(actuals))


# =========================================================================
# 9. NUMERICAL HEALTH CHECK
# =========================================================================

prediction_nan = np.isnan(
    predictions
).sum()

prediction_inf = np.isinf(
    predictions
).sum()

actual_nan = np.isnan(
    actuals
).sum()

actual_inf = np.isinf(
    actuals
).sum()

print("\n" + "=" * 100)
print("NUMERICAL HEALTH CHECK")
print("=" * 100)

print(
    "Prediction NaN:",
    prediction_nan
)

print(
    "Prediction Inf:",
    prediction_inf
)

print(
    "Actual NaN:",
    actual_nan
)

print(
    "Actual Inf:",
    actual_inf
)


# =========================================================================
# 10. VALID OBSERVATIONS
# =========================================================================

valid_mask = (
    np.isfinite(predictions)
    &
    np.isfinite(actuals)
)

valid_predictions = predictions[
    valid_mask
]

valid_actuals = actuals[
    valid_mask
]

print(
    "\nValid observations:",
    len(valid_predictions)
)

print(
    "Invalid observations:",
    (~valid_mask).sum()
)


# =========================================================================
# 11. ACTUAL DISTRIBUTION
# =========================================================================

print("\n" + "=" * 100)
print("ACTUAL RETURN DISTRIBUTION")
print("=" * 100)

print(
    pd.Series(
        valid_actuals
    ).describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)


# =========================================================================
# 12. PREDICTED DISTRIBUTION
# =========================================================================

print("\n" + "=" * 100)
print("PREDICTED RETURN DISTRIBUTION")
print("=" * 100)

print(
    pd.Series(
        valid_predictions
    ).describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)


# =========================================================================
# 13. MAE / RMSE
# =========================================================================

mae = mean_absolute_error(
    valid_actuals,
    valid_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        valid_actuals,
        valid_predictions
    )
)


# =========================================================================
# 14. DIRECTIONAL ACCURACY
# =========================================================================

actual_direction = np.sign(
    valid_actuals
)

prediction_direction = np.sign(
    valid_predictions
)

directional_accuracy = (
    np.mean(
        actual_direction
        ==
        prediction_direction
    )
    * 100
)


nonzero_mask = (
    valid_actuals != 0
)

if nonzero_mask.sum() > 0:

    directional_accuracy_nonzero = (
        np.mean(
            np.sign(
                valid_actuals[
                    nonzero_mask
                ]
            )
            ==
            np.sign(
                valid_predictions[
                    nonzero_mask
                ]
            )
        )
        * 100
    )

else:

    directional_accuracy_nonzero = np.nan


# =========================================================================
# 15. METRICS
# =========================================================================

print("\n" + "=" * 100)
print("VALIDATION METRICS")
print("=" * 100)

print(
    f"\nMAE                  : "
    f"{mae:.6f}"
)

print(
    f"RMSE                 : "
    f"{rmse:.6f}"
)

print(
    f"Directional Accuracy : "
    f"{directional_accuracy:.2f}%"
)

print(
    f"Directional Accuracy "
    f"(non-zero actuals)  : "
    f"{directional_accuracy_nonzero:.2f}%"
)


# =========================================================================
# 16. ERROR DISTRIBUTION
# =========================================================================

errors = (
    valid_predictions
    -
    valid_actuals
)

absolute_errors = np.abs(
    errors
)

print("\n" + "=" * 100)
print("ABSOLUTE ERROR DISTRIBUTION")
print("=" * 100)

print(
    pd.Series(
        absolute_errors
    ).describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# =========================================================================
# 17. TOP 20 LARGEST ERRORS
# =========================================================================

evaluation_df = pd.DataFrame({

    "actual_return":
        valid_actuals,

    "predicted_return":
        valid_predictions,

    "error":
        valid_predictions
        -
        valid_actuals,

    "absolute_error":
        np.abs(
            valid_predictions
            -
            valid_actuals
        )
})

evaluation_df = (
    evaluation_df
    .sort_values(
        "absolute_error",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 100)
print("TOP 20 LARGEST VALIDATION ERRORS")
print("=" * 100)

print(
    evaluation_df
    .head(20)
    .to_string(index=False)
)


# =========================================================================
# 18. EXTREME RETURNS
# =========================================================================

extreme_mask = (
    np.abs(valid_actuals) > 50
)

print("\n" + "=" * 100)
print("EXTREME ACTUAL RETURNS")
print("=" * 100)

print(
    "Observations with "
    "|actual return| > 50%:",
    extreme_mask.sum()
)

if extreme_mask.sum() > 0:

    extreme_df = pd.DataFrame({

        "actual_return":
            valid_actuals[
                extreme_mask
            ],

        "predicted_return":
            valid_predictions[
                extreme_mask
            ]
    })

    extreme_df["error"] = (
        extreme_df["predicted_return"]
        -
        extreme_df["actual_return"]
    )

    extreme_df["absolute_error"] = (
        np.abs(
            extreme_df["error"]
        )
    )

    print(
        extreme_df
        .sort_values(
            "absolute_error",
            ascending=False
        )
        .head(20)
        .to_string(index=False)
    )


# =========================================================================
# 19. SAVE RESULTS
# =========================================================================

evaluation_output = os.path.join(
    r"C:\Users\307124\Documents"
    r"\Mutual-Fund-Disclosure-Intelligence",
    "data",
    "processed",
    "tft_validation_predictions.csv"
)

evaluation_df.to_csv(
    evaluation_output,
    index=False
)

print("\n" + "=" * 100)
print("EVALUATION COMPLETE")
print("=" * 100)

print(
    "\nSaved validation predictions:"
)

print(
    evaluation_output
)

print("\n✅ Cell 35 completed successfully.")

TFT CHECKPOINT VALIDATION

Available checkpoints:
 - tft-best-epoch=00-val_loss=211653.8906.ckpt | val_loss=211653.8906
 - tft-best-epoch=01-val_loss=277781.7812.ckpt | val_loss=277781.7812

----------------------------------------------------------------------------------------------------
BEST CHECKPOINT
----------------------------------------------------------------------------------------------------
File:
tft-best-epoch=00-val_loss=211653.8906.ckpt
Validation loss: 211653.890600

Loading checkpoint...
✅ Checkpoint loaded.
Trainable parameters: 353661

Rebuilding validation dataset from training_dataset...
✅ Validation dataset rebuilt.
Validation samples: 14400
Validation batches: 113

Checking categorical indices...
encoder_cat shape: torch.Size([128, 24, 1])
encoder_cat min: 0
encoder_cat max: 4

Checking TFT categorical embedding...
Could not directly inspect ISIN embedding:
'Embedding' object has no attribute 'embedding'

RUNNING FULL VALIDATION PREDICTION
Processed 25/113
Pro

IndexError: index out of range in self